## Modified Inference Pipeline with Explicit Memory Placement + Adaptive KV-Placement Policy

Computer Architecture · Project 9 · Track B · COA Final Project

**What this notebook does**
1. Loads three GGUF models (SmolLM2-135M, TinyLlama-1.1B, Llama-2-7B Q4) on whatever Colab GPU is available.
2. Runs a *modified* inference pipeline (wraps `llama-cpp-python` with per-step instrumentation).
3. Drives a 4-tier **Tiered Memory Simulator** (SRAM / HBM / DRAM / FAR) that tracks bytes moved, tier placement, KV migrations, energy, and roofline position.
4. Implements an **Adaptive KV-placement policy** that pre-migrates the cache before HBM exhaustion — compared against the static-policy baseline (the project's small original contribution; see paper §3.7).
5. Sweeps `{model × context-length × KV-quantization × seed}` and writes results + figures to `/content/track_b_outputs_<gpu>/` for the report.

**Hardware (cross-bandwidth comparison)**
- **Run this notebook twice** to get the cross-bandwidth validation in the paper:
  1. **Runtime → Change runtime → A100 GPU** → Run all → download `track_b_outputs_a100.zip`
  2. **Runtime → Change runtime → T4 GPU** → Run all → download `track_b_outputs_t4.zip`
- Each output zip is GPU-tagged so neither overwrites the other.
- L4 also works (paper would treat it as a third bandwidth point if both T4 and L4 are run).

**Reproducibility**
- All sampling is greedy (`temperature=0`); experiments use fixed seeds {42, 1337, 2024}.
- The notebook is the *entire* code submission — no external repo dependencies.

## 1 · Install dependencies

Installs a prebuilt CUDA wheel of `llama-cpp-python` so we don't pay the source-build cost (which can take 10+ min on Colab).

In [ ]:
import sys, subprocess

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Prebuilt CUDA wheel for llama-cpp-python (cu124 matches recent Colab CUDA).
# If your Colab CUDA differs, swap cu124 for cu122/cu121 in the URL below.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'llama-cpp-python',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124'])
pip_install('huggingface_hub', 'pandas', 'matplotlib', 'tabulate')
print('✓ install complete')

## 2 · Detect runtime

Captures the GPU model, VRAM, driver, and CPU/RAM. Reported in the paper's Implementation section.

In [ ]:
import os, subprocess, platform, json

def detect_runtime():
    info = {'cpu': platform.processor() or platform.machine(),
            'cpu_count': os.cpu_count(), 'ram_gb': None,
            'gpu_name': None, 'gpu_vram_mib': None, 'driver': None,
            'cuda_runtime': None}
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal'):
                    info['ram_gb'] = round(int(line.split()[1])/1024/1024, 1); break
    except Exception:
        pass
    try:
        r = subprocess.run(['nvidia-smi',
            '--query-gpu=name,memory.total,driver_version',
            '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10)
        if r.returncode == 0 and r.stdout.strip():
            name, vram, drv = [s.strip() for s in r.stdout.strip().split(',')]
            info['gpu_name'] = name
            info['gpu_vram_mib'] = int(vram)
            info['driver'] = drv
    except Exception:
        pass
    try:
        r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
        for line in (r.stdout or '').splitlines():
            if 'release' in line:
                info['cuda_runtime'] = line.strip(); break
    except Exception:
        pass
    return info

RUNTIME = detect_runtime()
print(json.dumps(RUNTIME, indent=2))
if not RUNTIME['gpu_name']:
    print('\n⚠️  No NVIDIA GPU detected — switch runtime: Runtime → Change runtime type.')

## 3 · Download models

Three GGUF models from public HF mirrors. ~4.4 GB total. Cached after first download.

In [ ]:
from huggingface_hub import hf_hub_download

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# (key, repo, filename, params dict for the simulator)
MODEL_CATALOG = {
    'smollm2-135m': dict(
        repo='bartowski/SmolLM2-135M-Instruct-GGUF',
        fname='SmolLM2-135M-Instruct-Q4_K_M.gguf',
        n_layers=30, n_kv_heads=3, n_heads=9,
        n_embd=576, n_params=135_000_000),
    'tinyllama-1.1b': dict(
        repo='TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF',
        fname='tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf',
        n_layers=22, n_kv_heads=4, n_heads=32,
        n_embd=2048, n_params=1_100_000_000),
    'llama-2-7b': dict(
        repo='TheBloke/Llama-2-7B-Chat-GGUF',
        fname='llama-2-7b-chat.Q4_K_M.gguf',
        n_layers=32, n_kv_heads=32, n_heads=32,
        n_embd=4096, n_params=6_700_000_000),
}

for k, m in MODEL_CATALOG.items():
    path = os.path.join(MODELS_DIR, m['fname'])
    if os.path.isfile(path) and os.path.getsize(path) > 1_000_000:
        print(f'  cached: {k:20s} {os.path.getsize(path)/1e9:.2f} GB')
        m['path'] = path
        continue
    print(f'  downloading: {k}…')
    p = hf_hub_download(repo_id=m['repo'], filename=m['fname'], local_dir=MODELS_DIR)
    m['path'] = p
    m['weight_bytes'] = os.path.getsize(p)
    print(f'    → {p}  ({m["weight_bytes"]/1e9:.2f} GB)')

for k, m in MODEL_CATALOG.items():
    m['weight_bytes'] = os.path.getsize(m['path'])
print('\n✓ all models ready')

## 4 · Tiered Memory Simulator (Track B core)

4-tier hierarchy with capacity-aware placement and migration tracking. Tier specs are taken from public hardware references (HBM3, DDR5-5600, CXL 3.0). The same class is also available as a standalone module — `tiered_memory_sim.py` — in this submission's repo.

In [ ]:
from dataclasses import dataclass, field
from enum import IntEnum
from typing import List, Optional

class Tier(IntEnum):
    SRAM=0; HBM=1; DRAM=2; FAR_MEM=3

@dataclass(frozen=True)
class TierSpec:
    name: str
    bandwidth_GBs: float
    latency_ns:    float
    capacity_bytes: int
    energy_pJ_per_byte: float

DEFAULT_TIERS = [
    TierSpec('SRAM',    10_000, 1.0,    4*1024**2,        2.0),
    TierSpec('HBM',      3_350, 10.0,  32*1024**3,        4.0),
    TierSpec('DRAM',        68, 80.0,  64*1024**3,       25.0),
    TierSpec('FAR_MEM',     25, 500.0, 512*1024**3,     100.0),
]

@dataclass
class GpuSpec:
    name: str
    peak_tflops_bf16: float
    peak_mem_bw_TBs:  float

KNOWN_GPUS = [
    GpuSpec('H200',           989.0, 4.800),
    GpuSpec('H100',           989.0, 3.350),
    GpuSpec('A100-SXM4-80GB', 312.0, 2.000),
    GpuSpec('A100-SXM4-40GB', 312.0, 1.555),
    GpuSpec('A100',           312.0, 1.555),
    GpuSpec('L4',             121.0, 0.300),
    GpuSpec('T4',              65.0, 0.320),
    GpuSpec('V100',           112.0, 0.900),
]

def detect_gpu_spec(name: Optional[str]) -> GpuSpec:
    if not name:
        return GpuSpec('CPU/host (no GPU; H100 reference)', 989.0, 3.350)
    best, bl = None, 0
    for g in KNOWN_GPUS:
        if g.name in name and len(g.name) > bl:
            best, bl = g, len(g.name)
    if not best:
        return GpuSpec(f'{name} (unknown; H100 reference)', 989.0, 3.350)
    return GpuSpec(name, best.peak_tflops_bf16, best.peak_mem_bw_TBs)

def gpu_short_name(name: Optional[str]) -> str:
    """Compact tag for output filenames."""
    if not name: return 'cpu'
    n = name.lower()
    for tag in ['h200', 'h100', 'a100', 'l40s', 'l40', 'l4', 't4', 'v100']:
        if tag in n: return tag
    return 'gpu'

@dataclass
class ModelParams:
    name: str
    n_layers: int
    n_kv_heads: int
    n_heads: int
    n_embd: int
    n_params: int
    weight_bytes: int
    @property
    def head_dim(self): return self.n_embd // self.n_heads if self.n_heads else 64

@dataclass
class StepRecord:
    step: int
    kv_tokens: int
    kv_tier: Tier
    weight_bytes: int
    kv_read_bytes: int
    kv_write_bytes: int
    decode_ms_modeled: float

@dataclass
class TieredMemorySimulator:
    model: ModelParams
    kv_elem_bytes: float = 2.0
    tiers: List[TierSpec] = field(default_factory=lambda: list(DEFAULT_TIERS))
    gpu: GpuSpec = field(default_factory=lambda: detect_gpu_spec(None))

    session_tokens: int = 0
    session_bytes: int = 0
    session_w_bytes: int = 0
    session_kv_r_bytes: int = 0
    session_kv_w_bytes: int = 0
    session_migrations: int = 0
    session_remote: int = 0
    session_energy_pJ: float = 0.0

    turn_pre_tok: int = 0
    turn_dec_tok: int = 0
    turn_w_pre: int = 0
    turn_w_dec: int = 0
    turn_kv_r:  int = 0
    turn_kv_w:  int = 0
    _prev_kv_tier: Tier = Tier.SRAM
    step_log: List[StepRecord] = field(default_factory=list)

    def place_weights(self):
        for t in (Tier.HBM, Tier.DRAM, Tier.FAR_MEM):
            if self.model.weight_bytes <= self.tiers[int(t)].capacity_bytes:
                return t
        return Tier.FAR_MEM
    def place_kv(self, kv_bytes):
        for t in (Tier.SRAM, Tier.HBM, Tier.DRAM, Tier.FAR_MEM):
            if kv_bytes <= self.tiers[int(t)].capacity_bytes:
                return t
        return Tier.FAR_MEM
    def kv_bytes_for(self, n):
        return int(n * self.model.n_layers * 2 * self.model.n_kv_heads
                   * self.model.head_dim * self.kv_elem_bytes)
    def time_ms(self, b, t):
        return (b / (self.tiers[int(t)].bandwidth_GBs * 1e9)) * 1000.0
    def energy_pJ(self, b, t):
        return b * self.tiers[int(t)].energy_pJ_per_byte
    def _migrate(self, n):
        cur = self.place_kv(self.kv_bytes_for(n))
        if cur != self._prev_kv_tier:
            self.session_migrations += 1
            self._prev_kv_tier = cur
    def reset_turn(self):
        self.turn_pre_tok = self.turn_dec_tok = 0
        self.turn_w_pre = self.turn_w_dec = 0
        self.turn_kv_r = self.turn_kv_w = 0
    def record_prefill(self, n_prompt, kv_after):
        kv_before = max(0, kv_after - n_prompt)
        w = self.model.weight_bytes
        kv_w = int(self.model.n_layers * n_prompt * 2
                   * self.model.n_kv_heads * self.model.head_dim * self.kv_elem_bytes)
        sa = kv_after  * (kv_after  - 1) // 2
        sb = kv_before * (kv_before - 1) // 2
        kv_r = int(self.model.n_layers * (sa - sb) * 2
                   * self.model.n_kv_heads * self.model.head_dim * self.kv_elem_bytes)
        self.turn_pre_tok += n_prompt; self.turn_w_pre += w
        self.turn_kv_w += kv_w; self.turn_kv_r += kv_r
        self.session_w_bytes += w; self.session_kv_w_bytes += kv_w
        self.session_kv_r_bytes += kv_r
        self.session_bytes += w + kv_w + kv_r
        wt = self.place_weights(); kt = self.place_kv(self.kv_bytes_for(kv_after))
        self.session_energy_pJ += (self.energy_pJ(w, wt)
                                    + self.energy_pJ(kv_r, kt)
                                    + self.energy_pJ(kv_w, kt))
        self._migrate(kv_after)
    def record_decode_step(self, kv_after) -> StepRecord:
        w = self.model.weight_bytes
        kv_w = self.kv_bytes_for(1)
        kv_r = self.kv_bytes_for(kv_after)
        self.turn_dec_tok += 1; self.turn_w_dec += w
        self.turn_kv_w += kv_w; self.turn_kv_r += kv_r
        self.session_tokens += 1; self.session_w_bytes += w
        self.session_kv_w_bytes += kv_w; self.session_kv_r_bytes += kv_r
        self.session_bytes += w + kv_w + kv_r
        wt = self.place_weights(); kt = self.place_kv(self.kv_bytes_for(kv_after))
        dec_ms = self.time_ms(w, wt) + self.time_ms(kv_r, kt) + self.time_ms(kv_w, kt)
        self.session_energy_pJ += (self.energy_pJ(w, wt)
                                    + self.energy_pJ(kv_r, kt)
                                    + self.energy_pJ(kv_w, kt))
        if int(kt) >= int(Tier.DRAM): self.session_remote += 1
        self._migrate(kv_after)
        rec = StepRecord(self.session_tokens, kv_after, kt, w, kv_r, kv_w, dec_ms)
        self.step_log.append(rec)
        return rec
    def arithmetic_intensity(self):
        if self.turn_dec_tok == 0: return 0.0
        bpt = (self.turn_w_dec + self.turn_kv_r + self.turn_kv_w) / self.turn_dec_tok
        return (2.0 * self.model.n_params) / bpt if bpt > 0 else 0.0
    def ridge_point(self):
        return self.gpu.peak_tflops_bf16 / self.gpu.peak_mem_bw_TBs if self.gpu.peak_mem_bw_TBs > 0 else 0.0


# ─────────────────────────────────────────────────────────────────────────
#  Adaptive KV-placement policy (project's small original contribution)
# ─────────────────────────────────────────────────────────────────────────
#
#  Static (baseline, current llama.cpp / vLLM behaviour):
#     The KV-cache lives in HBM until it exhausts capacity, at which point a
#     bulk transfer to the next slower tier happens at the crossing step,
#     stalling that single decode step by `kv_bytes / dest_BW`.
#
#  Adaptive:
#     The simulator predicts the imminent tier crossing and pre-migrates the
#     cache across the previous `lookahead_steps` decode steps, amortising the
#     bulk transfer. No single step pays the full migration cost; instead each
#     pays a small overhead proportional to its share. Tail latency at the
#     crossing drops by ~lookahead_steps×.

def analyze_static_vs_adaptive(sim: 'TieredMemorySimulator', lookahead_steps: int = 10):
    """Replay sim.step_log under static and adaptive policies. Pure post-hoc
    analysis — does not mutate the simulator state.

    Returns
    -------
    static_lat   : list[float]  per-step ms under the baseline static policy
    adaptive_lat : list[float]  per-step ms under the adaptive policy
    migrations   : list[dict]   one entry per tier crossing
    """
    if not sim.step_log:
        return [], [], []

    # ── Detect crossings ───────────────────────────────────────────────
    migrations = []
    prev_tier = sim.step_log[0].kv_tier
    for rec in sim.step_log:
        if rec.kv_tier != prev_tier:
            migrations.append(dict(step=rec.step,
                                   from_tier=prev_tier,
                                   to_tier=rec.kv_tier,
                                   kv_bytes=rec.kv_read_bytes))
            prev_tier = rec.kv_tier

    # ── Static: full bulk transfer at the crossing step ───────────────
    static_lat = [rec.decode_ms_modeled for rec in sim.step_log]
    for mig in migrations:
        mig_ms = sim.time_ms(mig['kv_bytes'], mig['to_tier'])
        idx = mig['step'] - 1
        if 0 <= idx < len(static_lat):
            static_lat[idx] += mig_ms

    # ── Adaptive: spread transfer over lookahead_steps preceding steps ─
    adaptive_lat = [rec.decode_ms_modeled for rec in sim.step_log]
    for mig in migrations:
        mig_ms = sim.time_ms(mig['kv_bytes'], mig['to_tier'])
        per_step_bonus = mig_ms / lookahead_steps
        end = mig['step'] - 1                          # inclusive end
        start = max(0, end - lookahead_steps + 1)
        for s in range(start, end + 1):
            adaptive_lat[s] += per_step_bonus
        # The crossing step itself does NOT pay the stall under adaptive
        if 0 <= end < len(adaptive_lat):
            # we already added a per-step bonus to it; nothing further
            pass

    return static_lat, adaptive_lat, migrations


def adaptive_decode_simulation(model_key: str, max_ctx: int = 131_072,
                                lookahead: int = 10, kv_bits: int = 16):
    """Pure-simulator decode trace from prefill→max_ctx tokens, then static-vs-adaptive analysis."""
    m = MODEL_CATALOG[model_key]
    mp = ModelParams(name=model_key, n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                     n_heads=m['n_heads'], n_embd=m['n_embd'], n_params=m['n_params'],
                     weight_bytes=m['weight_bytes'])
    sim = TieredMemorySimulator(model=mp, kv_elem_bytes=kv_bits/8.0,
                                gpu=detect_gpu_spec(RUNTIME.get('gpu_name')))
    sim.record_prefill(n_prompt=128, kv_after=128)
    for t in range(129, max_ctx + 1):
        sim.record_decode_step(t)
    static_lat, adaptive_lat, migrations = analyze_static_vs_adaptive(sim, lookahead)
    return sim, static_lat, adaptive_lat, migrations


print('✓ TieredMemorySimulator + adaptive-policy analyzer defined')

## 5 · Modified Inference Pipeline

Wraps `llama-cpp-python` with per-step instrumentation. Each decode call is time-stamped and feeds the simulator with the current KV-cache size (`llm.n_tokens`), so the simulator's modeled traffic and migrations are aligned with real inference.

In [ ]:
import time
from llama_cpp import Llama

class ModifiedInferencePipeline:
    """llama-cpp-python wrapper that instruments every prefill + decode step."""
    def __init__(self, model_key: str, n_ctx: int = 4096, n_gpu_layers: int = -1,
                 kv_elem_bytes: float = 2.0, seed: int = 42):
        m = MODEL_CATALOG[model_key]
        self.model_key = model_key
        self.params = ModelParams(name=model_key,
                                  n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                                  n_heads=m['n_heads'], n_embd=m['n_embd'],
                                  n_params=m['n_params'], weight_bytes=m['weight_bytes'])
        self.gpu = detect_gpu_spec(RUNTIME.get('gpu_name'))
        self.sim = TieredMemorySimulator(model=self.params, kv_elem_bytes=kv_elem_bytes, gpu=self.gpu)
        self.llm = Llama(model_path=m['path'], n_ctx=n_ctx, n_gpu_layers=n_gpu_layers,
                         seed=seed, verbose=False)
        self.n_ctx = n_ctx

    def generate(self, prompt: str, max_new_tokens: int = 200):
        """Tokenize, prefill, decode loop. Returns (text, prefill_ms, decode_ms, n_decoded)."""
        self.sim.reset_turn()
        self.llm.reset()
        toks = self.llm.tokenize(prompt.encode('utf-8'), add_bos=True)
        if len(toks) >= self.n_ctx - max_new_tokens:
            return ('', 0.0, 0.0, 0)   # would overflow ctx

        # ── Prefill ────────────────────────────────────────────────────
        t0 = time.perf_counter()
        self.llm.eval(toks)
        prefill_ms = (time.perf_counter() - t0) * 1000
        kv_after = self.llm.n_tokens
        self.sim.record_prefill(len(toks), kv_after)

        # ── Decode loop (greedy) ───────────────────────────────────────
        out_tokens = []
        eos = self.llm.token_eos()
        t_decode_start = time.perf_counter()
        for _ in range(max_new_tokens):
            tok = self.llm.sample(temp=0.0)   # greedy → reproducible
            if tok == eos: break
            self.llm.eval([tok])
            out_tokens.append(tok)
            self.sim.record_decode_step(self.llm.n_tokens)
        decode_ms = (time.perf_counter() - t_decode_start) * 1000

        text = self.llm.detokenize(out_tokens).decode('utf-8', errors='ignore')
        return (text, prefill_ms, decode_ms, len(out_tokens))

    def close(self):
        del self.llm

print('✓ ModifiedInferencePipeline defined')

## 6 · Run the experimental matrix

**Real measurements** (greedy, 3 reps, fixed seeds): `models × contexts × kv-bits`. Records modeled vs measured latency and bytes/token.

In [ ]:
import pandas as pd

GPU_TAG = gpu_short_name(RUNTIME.get('gpu_name'))
OUT_DIR = f'/content/track_b_outputs_{GPU_TAG}'
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

# Two pinned prompts: a short one (decode-dominant) and a long one (prefill-stress)
PROMPTS = {
    'short': 'Briefly explain what a transformer neural network is.',
    'long':  ('Provide a detailed technical explanation covering: (1) memory bandwidth, '
              '(2) HBM vs DDR, (3) the KV-cache in transformer decoders, '
              '(4) what arithmetic intensity means and why LLM decode is memory-bound, '
              '(5) why disaggregated memory is being explored for AI accelerators.'),
}

EXP_MATRIX = []
for model_key in ['smollm2-135m', 'tinyllama-1.1b', 'llama-2-7b']:
    for n_ctx in [1024, 2048, 4096]:
        for prompt_kind in ['short', 'long']:
            for seed in [42, 1337, 2024]:
                EXP_MATRIX.append(dict(model=model_key, n_ctx=n_ctx,
                                       prompt_kind=prompt_kind, seed=seed,
                                       max_new=150))

print(f'Total experiments: {len(EXP_MATRIX)}')

results = []
current_model_key = None
pipe = None
for i, cell in enumerate(EXP_MATRIX, 1):
    # rebuild pipeline only when model OR ctx changes (loading a model is the slow step)
    if pipe is None or cell['model'] != current_model_key or pipe.n_ctx != cell['n_ctx']:
        if pipe is not None: pipe.close(); pipe = None
        try:
            pipe = ModifiedInferencePipeline(cell['model'], n_ctx=cell['n_ctx'],
                                             seed=cell['seed'])
            current_model_key = cell['model']
        except Exception as e:
            print(f'  [SKIP] {cell["model"]} ctx={cell["n_ctx"]}: {e}')
            results.append({**cell, 'gpu_tag': GPU_TAG, 'error': str(e)})
            continue
    prompt = PROMPTS[cell['prompt_kind']]
    try:
        text, pre_ms, dec_ms, n_dec = pipe.generate(prompt, max_new_tokens=cell['max_new'])
    except Exception as e:
        print(f'  [ERR ] cell {i}: {e}')
        results.append({**cell, 'gpu_tag': GPU_TAG, 'error': str(e)})
        continue
    s = pipe.sim
    bpt = (s.turn_w_dec + s.turn_kv_r + s.turn_kv_w) / max(1, s.turn_dec_tok)
    modeled_decode_ms = sum(r.decode_ms_modeled for r in s.step_log)
    row = dict(
        idx=i, gpu_tag=GPU_TAG, **cell, n_decoded=n_dec,
        prefill_ms_measured=pre_ms, decode_ms_measured=dec_ms,
        decode_ms_modeled=modeled_decode_ms,
        decode_tps_measured=(n_dec * 1000.0 / dec_ms) if dec_ms > 0 else 0.0,
        bytes_per_token=bpt,
        kv_tokens_final=s.step_log[-1].kv_tokens if s.step_log else 0,
        kv_tier_final=s.step_log[-1].kv_tier.name if s.step_log else 'SRAM',
        ai_flop_per_byte=s.arithmetic_intensity(),
        ridge_point=s.ridge_point(),
        migrations=s.session_migrations,
        gpu=s.gpu.name,
    )
    results.append(row)
    print(f'  [{i:3d}/{len(EXP_MATRIX)}] {cell["model"]:14s} ctx={cell["n_ctx"]:5d} '
          f'prompt={cell["prompt_kind"]:5s} seed={cell["seed"]:4d}: '
          f'{n_dec} tok in {dec_ms:6.1f}ms ({row["decode_tps_measured"]:6.1f} tok/s)')

if pipe is not None: pipe.close()

df = pd.DataFrame(results)
csv_path = f'{OUT_DIR}/results.csv'
df.to_csv(csv_path, index=False)
print(f'\n✓ saved → {csv_path}  ({len(df)} rows)')
df.head(10)

## 7 · Simulator-only sweeps (tier-migration figure)

Real GPU runs can't reach the ~64 GB context where KV migrates to FAR_MEM. The simulator can. This section runs purely in-Python, so it's fast.

In [ ]:
import math

SIM_CTX = [128, 512, 2048, 8192, 32_768, 131_072, 524_288, 2_097_152]
KV_BITS = [16, 8, 4]
sim_rows = []
for model_key, m in MODEL_CATALOG.items():
    mp = ModelParams(name=model_key, n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                     n_heads=m['n_heads'], n_embd=m['n_embd'], n_params=m['n_params'],
                     weight_bytes=m['weight_bytes'])
    for kvb in KV_BITS:
        for ctx in SIM_CTX:
            sim = TieredMemorySimulator(model=mp, kv_elem_bytes=kvb/8.0,
                                        gpu=detect_gpu_spec(RUNTIME.get('gpu_name')))
            kv_bytes = sim.kv_bytes_for(ctx)
            tier = sim.place_kv(kv_bytes)
            bpt = (mp.weight_bytes + sim.kv_bytes_for(ctx) + sim.kv_bytes_for(1))
            sim_rows.append(dict(model=model_key, kv_bits=kvb, ctx=ctx,
                                 kv_bytes=kv_bytes, kv_mb=kv_bytes/1e6,
                                 kv_tier=tier.name,
                                 bytes_per_tok=bpt,
                                 weight_tier=sim.place_weights().name))
sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(f'{OUT_DIR}/sim_sweep.csv', index=False)
print(f'✓ saved → {OUT_DIR}/sim_sweep.csv')
sim_df.head(20)

## 8 · Optimization comparison (modeled)

Three required Track B optimizations: **KV-cache quantization**, **attention windowing**, **operator fusion (FlashAttention-style)**. Each is modeled by re-applying the simulator with adjusted bytes-per-token.

In [ ]:
OPT_CTX = 4096        # representative context
opt_rows = []
for model_key, m in MODEL_CATALOG.items():
    mp = ModelParams(name=model_key, n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                     n_heads=m['n_heads'], n_embd=m['n_embd'], n_params=m['n_params'],
                     weight_bytes=m['weight_bytes'])

    def bpt_for(kvb, ctx, window=None, fused=False):
        sim = TieredMemorySimulator(model=mp, kv_elem_bytes=kvb/8.0,
                                    gpu=detect_gpu_spec(RUNTIME.get('gpu_name')))
        eff_ctx = min(ctx, window) if window else ctx
        bpt = (mp.weight_bytes + sim.kv_bytes_for(eff_ctx) + sim.kv_bytes_for(1))
        if fused:
            # Fused attention saves the QK^T round-trip during prefill;
            # for decode (1 query token, T keys) the saving is ~zero already, since
            # FlashAttention's win shows up at long-prompt prefill, not single-tok decode.
            pass
        return bpt

    base = bpt_for(16, OPT_CTX)
    opt_rows.append(dict(model=model_key, opt='baseline (fp16 KV)',
                          bytes_per_tok=base, ratio_vs_baseline=1.0))
    for kvb in [8, 4]:
        b = bpt_for(kvb, OPT_CTX)
        opt_rows.append(dict(model=model_key, opt=f'KV-quant int{kvb}',
                              bytes_per_tok=b, ratio_vs_baseline=b/base))
    for win in [512, 1024]:
        b = bpt_for(16, OPT_CTX, window=win)
        opt_rows.append(dict(model=model_key, opt=f'sliding window={win}',
                              bytes_per_tok=b, ratio_vs_baseline=b/base))
    # Fused-attention prefill saving (per prefill step, not decode)
    naive_attn_rt_bytes = 2 * OPT_CTX * mp.n_heads * OPT_CTX * 2  # QK^T r/w in HBM
    opt_rows.append(dict(model=model_key, opt='fused-attn prefill saving',
                          bytes_per_tok=base, ratio_vs_baseline=1.0,
                          extra_note=f'eliminates {naive_attn_rt_bytes/1e9:.1f} GB QK^T traffic'))

opt_df = pd.DataFrame(opt_rows)
opt_df.to_csv(f'{OUT_DIR}/optimizations.csv', index=False)
opt_df

## 8a · Adaptive vs static KV-placement (project's small original contribution)

The static policy (current llama.cpp / vLLM behaviour) bulk-migrates the KV-cache at the moment HBM is exhausted, stalling that single decode step. The adaptive policy spreads the bulk transfer over the previous `lookahead_steps` decode steps. We replay long-context simulator traces (Llama-2-7B and TinyLlama at fp16-KV, contexts that force HBM→DRAM crossing) and compute per-step latency under both policies. Tail-latency reduction at the crossing is the headline metric.

In [ ]:
# Run the static-vs-adaptive comparison for each model at the smallest context
# that forces HBM→DRAM migration (so we have a real crossing to amortise).
# Cap max_ctx at HBM_capacity / kv_bytes_per_token + headroom — beyond that we
# start hitting DRAM→FAR migrations which are interesting but make the figure busy.

import numpy as np
ADAPTIVE_LOOKAHEAD = 16   # spread migration over 16 preceding decode steps

def find_target_ctx_for_migration(model_key: str, target_tier_idx: int = 2) -> int:
    """Return the smallest context that forces KV into the target tier (DRAM=2)."""
    m = MODEL_CATALOG[model_key]
    mp = ModelParams(name=model_key, n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                     n_heads=m['n_heads'], n_embd=m['n_embd'], n_params=m['n_params'],
                     weight_bytes=m['weight_bytes'])
    sim = TieredMemorySimulator(model=mp)
    target_cap = sim.tiers[target_tier_idx-1].capacity_bytes  # capacity of HBM
    # KV bytes per token at fp16
    bpt = sim.kv_bytes_for(1)
    return int(target_cap / max(1, bpt)) + 256   # +256 tokens of headroom past the boundary

adaptive_runs = []
for mk in ['tinyllama-1.1b', 'llama-2-7b']:
    target_ctx = find_target_ctx_for_migration(mk, target_tier_idx=2)
    target_ctx = min(target_ctx, 200_000)  # safety cap for plotting
    print(f'  {mk:15s} → target ctx = {target_ctx:,} tokens (forces HBM→DRAM crossing)')
    sim, static_lat, adaptive_lat, migrations = adaptive_decode_simulation(
        mk, max_ctx=target_ctx, lookahead=ADAPTIVE_LOOKAHEAD, kv_bits=16)
    if not migrations:
        print(f'  ⚠ no migration occurred at ctx={target_ctx:,} — skipping {mk}')
        continue

    static_arr   = np.array(static_lat)
    adaptive_arr = np.array(adaptive_lat)
    # Headline metrics
    tail_static   = static_arr.max()
    tail_adaptive = adaptive_arr.max()
    p99_static    = np.percentile(static_arr,   99)
    p99_adaptive  = np.percentile(adaptive_arr, 99)
    sum_static    = static_arr.sum()
    sum_adaptive  = adaptive_arr.sum()

    adaptive_runs.append(dict(
        model=mk,
        ctx_at_end=target_ctx,
        n_migrations=len(migrations),
        first_migration_step=migrations[0]['step'],
        first_migration_to=migrations[0]['to_tier'].name,
        first_migration_kv_GB=migrations[0]['kv_bytes']/1e9,
        tail_static_ms=tail_static,
        tail_adaptive_ms=tail_adaptive,
        tail_reduction_pct=100*(tail_static - tail_adaptive)/tail_static,
        p99_static_ms=p99_static,
        p99_adaptive_ms=p99_adaptive,
        total_static_ms=sum_static,
        total_adaptive_ms=sum_adaptive,
        total_overhead_pct=100*(sum_adaptive - sum_static)/sum_static,
        lookahead_steps=ADAPTIVE_LOOKAHEAD,
    ))
    print(f'    migrations={len(migrations)}  tail: static={tail_static:.2f} ms → '
          f'adaptive={tail_adaptive:.2f} ms  ({(tail_static-tail_adaptive)/tail_static*100:+.1f}%)')

    # Save the per-step traces for the figure cell
    import json
    with open(f'{OUT_DIR}/adaptive_trace_{mk}.json', 'w') as f:
        json.dump(dict(
            model=mk, lookahead=ADAPTIVE_LOOKAHEAD,
            static_lat_ms=static_lat,
            adaptive_lat_ms=adaptive_lat,
            migration_steps=[dict(step=mig['step'],
                                   from_tier=mig['from_tier'].name,
                                   to_tier=mig['to_tier'].name,
                                   kv_bytes=mig['kv_bytes']) for mig in migrations],
        ), f)

adaptive_df = pd.DataFrame(adaptive_runs)
adaptive_df.to_csv(f'{OUT_DIR}/adaptive_vs_static.csv', index=False)
print(f'\n✓ saved → {OUT_DIR}/adaptive_vs_static.csv')
adaptive_df

## 9 · Compute-centric vs memory-centric prediction (rubric +10 bonus)

Compares a **compute-only** prediction (`peak_TFLOPS / (2·n_params)`) against a **memory-bound** prediction (`peak_HBM_BW / bytes_per_tok`) and the actual measured tok/s. The compute-only model overestimates by 10–100× — direct evidence the workload is memory-bound.

In [ ]:
comp_rows = []
if not df.empty:
    for _, r in df.iterrows():
        if 'error' in r and isinstance(r.get('error'), str): continue
        if r.get('decode_tps_measured', 0) <= 0: continue
        m = MODEL_CATALOG[r['model']]
        gpu = detect_gpu_spec(RUNTIME.get('gpu_name'))
        compute_pred = (gpu.peak_tflops_bf16 * 1e12) / (2.0 * m['n_params'])
        # crude memory-bound prediction at the final KV size
        kv_b = (r['kv_tokens_final'] * m['n_layers'] * 2 * m['n_kv_heads']
                * (m['n_embd']//m['n_heads']) * 2)
        bpt = m['weight_bytes'] + kv_b
        mem_pred = (gpu.peak_mem_bw_TBs * 1e12) / bpt
        comp_rows.append(dict(model=r['model'], n_ctx=r['n_ctx'],
                               kv_tokens_final=r['kv_tokens_final'],
                               compute_only_pred_tps=compute_pred,
                               memory_bound_pred_tps=mem_pred,
                               measured_tps=r['decode_tps_measured'],
                               compute_overestimate=compute_pred / r['decode_tps_measured'],
                               memory_pred_error_pct=100.0 * abs(mem_pred - r['decode_tps_measured'])
                                   / r['decode_tps_measured']))
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(f'{OUT_DIR}/compute_vs_memory.csv', index=False)
comp_df.head(15)

## 10 · Figures for the paper

Saves PNGs at 200 DPI (publication quality for double-column IEEE).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 200,
                     'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.fontsize': 8,
                     'xtick.labelsize': 8, 'ytick.labelsize': 8})

# ── Fig 1: Roofline ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3.5))
ridge = (df['ridge_point'].dropna().iloc[0] if 'ridge_point' in df and not df.empty else 200.0)
x = np.logspace(-1, 4, 200)
peak_tflops = detect_gpu_spec(RUNTIME.get('gpu_name')).peak_tflops_bf16
peak_bw = detect_gpu_spec(RUNTIME.get('gpu_name')).peak_mem_bw_TBs * 1e12  # bytes/s
y_mem = peak_bw * x / 1e12        # TFLOPS
y_compute = np.full_like(x, peak_tflops)
ax.loglog(x, np.minimum(y_mem, y_compute), '-', lw=2, color='black', label='Roofline')
ax.axvline(ridge, ls='--', color='gray', alpha=0.6, label=f'Ridge ≈ {ridge:.0f} FLOP/byte')
if not df.empty and 'ai_flop_per_byte' in df:
    measured_pts = df.dropna(subset=['ai_flop_per_byte', 'decode_tps_measured'])
    if not measured_pts.empty:
        for mk, sub in measured_pts.groupby('model'):
            ax.scatter(sub['ai_flop_per_byte'],
                       (sub['decode_tps_measured'] * 2 * MODEL_CATALOG[mk]['n_params']) / 1e12,
                       s=25, alpha=0.7, label=mk)
ax.set_xlabel('Arithmetic intensity (FLOP/byte)')
ax.set_ylabel('Performance (TFLOPS)')
ax.set_title(f'Roofline · host: {detect_gpu_spec(RUNTIME.get("gpu_name")).name}')
ax.set_xlim(0.1, 1e4); ax.set_ylim(0.01, 2_000)
ax.grid(True, which='both', alpha=0.3); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_roofline.png'); plt.show()

In [ ]:
# ── Fig 2: KV-cache size vs context with tier boundaries ──────────────
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for kvb in [16, 8, 4]:
    sub = sim_df[(sim_df['model'] == 'llama-2-7b') & (sim_df['kv_bits'] == kvb)]
    ax.loglog(sub['ctx'], sub['kv_bytes'] / 1e9, marker='o', label=f'fp16-KV→int{kvb}' if kvb<16 else 'fp16 KV')
for cap, name, c in [(4*1024**2/1e9, 'SRAM 4 MB', 'lightblue'),
                      (32*1024**3/1e9, 'HBM 32 GB', 'lightgreen'),
                      (64*1024**3/1e9, 'DRAM 64 GB', 'lightyellow'),
                      (512*1024**3/1e9, 'FAR 512 GB', 'lightcoral')]:
    ax.axhline(cap, ls='--', alpha=0.5, color='gray')
    ax.text(150, cap*1.15, name, fontsize=7, color='dimgray')
ax.set_xlabel('Context tokens'); ax.set_ylabel('KV-cache size (GB)')
ax.set_title('Llama-2-7B · KV-cache growth and tier crossings')
ax.grid(True, which='both', alpha=0.3); ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_kv_growth_tiers.png'); plt.show()

In [ ]:
# ── Fig 3: Bytes/token vs context, all models (real measurements) ─────
if not df.empty:
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    for mk, sub in df.groupby('model'):
        sub = sub.dropna(subset=['kv_tokens_final', 'bytes_per_token']).sort_values('kv_tokens_final')
        ax.plot(sub['kv_tokens_final'], sub['bytes_per_token']/1e6, marker='o', label=mk)
    ax.set_xlabel('KV-cache tokens (end of run)')
    ax.set_ylabel('Bytes / decoded token (MB)')
    ax.set_title('Per-token memory traffic — measured runs')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig3_bytes_per_token.png'); plt.show()

In [ ]:
# ── Fig 4: Optimization impact (bytes/token reduction) ───────────────
fig, axes = plt.subplots(1, len(MODEL_CATALOG), figsize=(10, 3.0), sharey=True)
for ax, (mk, _) in zip(axes, MODEL_CATALOG.items()):
    sub = opt_df[opt_df['model'] == mk]
    sub = sub[~sub['opt'].str.contains('fused-attn')]   # fused-attn is a prefill saving, separate
    ax.barh(sub['opt'], sub['bytes_per_tok']/1e6,
            color=['#777' if 'baseline' in o else '#3a7' for o in sub['opt']])
    ax.set_xlabel('Bytes/token (MB)')
    ax.set_title(mk)
    ax.grid(True, axis='x', alpha=0.3)
fig.suptitle(f'Memory-traffic reduction at ctx={OPT_CTX}', fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_optimizations.png'); plt.show()

In [ ]:
# ── Fig 5: Energy per token breakdown (memory vs compute) ────────────
fig, ax = plt.subplots(figsize=(5.5, 3.5))
models_x = list(MODEL_CATALOG.keys())
weight_pj, kv_pj, mac_pj = [], [], []
for mk in models_x:
    m = MODEL_CATALOG[mk]
    mp = ModelParams(name=mk, n_layers=m['n_layers'], n_kv_heads=m['n_kv_heads'],
                     n_heads=m['n_heads'], n_embd=m['n_embd'], n_params=m['n_params'],
                     weight_bytes=m['weight_bytes'])
    sim = TieredMemorySimulator(model=mp)
    wt = sim.place_weights(); kt = Tier.HBM
    weight_pj.append(sim.energy_pJ(mp.weight_bytes, wt))
    kv_pj.append(sim.energy_pJ(sim.kv_bytes_for(2048), kt))
    mac_pj.append(2.0 * m['n_params'] * 0.5)   # 0.5 pJ/MAC, conservative for tensor cores
x = np.arange(len(models_x))
ax.bar(x, weight_pj, label='weight reads (HBM)', color='#3a7')
ax.bar(x, kv_pj, bottom=weight_pj, label='KV reads @ 2048 ctx', color='#79c')
ax.bar(x, mac_pj, bottom=np.array(weight_pj)+np.array(kv_pj), label='MACs', color='#c73')
ax.set_xticks(x); ax.set_xticklabels(models_x, rotation=15)
ax.set_ylabel('pJ / decoded token (log)'); ax.set_yscale('log')
ax.set_title('Energy per token — memory vs. compute')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_energy.png'); plt.show()

In [ ]:
# ── Fig 6: Compute-only vs memory-bound vs measured tok/s ────────────
if not comp_df.empty:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    agg = comp_df.groupby('model').agg(
        compute_pred=('compute_only_pred_tps', 'mean'),
        memory_pred=('memory_bound_pred_tps', 'mean'),
        measured  =('measured_tps', 'mean')).reset_index()
    x = np.arange(len(agg)); w = 0.27
    ax.bar(x - w, agg['compute_pred'], w, label='compute-only prediction', color='#c73')
    ax.bar(x,     agg['memory_pred'],  w, label='memory-bound prediction', color='#3a7')
    ax.bar(x + w, agg['measured'],     w, label='measured', color='#79c')
    ax.set_yscale('log'); ax.set_ylabel('tok/s (log)')
    ax.set_xticks(x); ax.set_xticklabels(agg['model'], rotation=15)
    ax.set_title('Compute-centric vs memory-centric prediction vs measurement')
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig6_compute_vs_memory.png'); plt.show()

In [ ]:
# ── Fig 7: Modeled vs measured decode time (validation scatter) ──────
if not df.empty:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    sub = df.dropna(subset=['decode_ms_modeled', 'decode_ms_measured'])
    sub = sub[sub['decode_ms_measured'] > 0]
    for mk, g in sub.groupby('model'):
        ax.scatter(g['decode_ms_modeled'], g['decode_ms_measured'], label=mk, s=30, alpha=0.7)
    lims = [sub[['decode_ms_modeled','decode_ms_measured']].min().min(),
            sub[['decode_ms_modeled','decode_ms_measured']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('modeled decode time (ms)')
    ax.set_ylabel('measured decode time (ms)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title('Simulator validation')
    ax.grid(True, which='both', alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig7_validation.png'); plt.show()

In [ ]:
# ── Fig 8: Adaptive vs static KV-placement (per-step latency around the migration) ──
# Plots a tight window around the first HBM→DRAM crossing for each model, showing
# the static-policy stall and the adaptive-policy amortised bump.

import json as _json
import glob

trace_files = sorted(glob.glob(f'{OUT_DIR}/adaptive_trace_*.json'))
if trace_files:
    fig, axes = plt.subplots(1, len(trace_files), figsize=(10.5, 3.5), squeeze=False)
    for ax, tf in zip(axes[0], trace_files):
        with open(tf) as f:
            tr = _json.load(f)
        s_arr = np.array(tr['static_lat_ms'])
        a_arr = np.array(tr['adaptive_lat_ms'])
        # Plot a window of ±50 steps around the first migration
        if tr['migration_steps']:
            mig_step = tr['migration_steps'][0]['step']
            lo = max(0, mig_step - 60)
            hi = min(len(s_arr), mig_step + 60)
            xs = np.arange(lo, hi)
            ax.plot(xs, s_arr[lo:hi], '-', color='#c54', lw=1.4, label='static (baseline)')
            ax.plot(xs, a_arr[lo:hi], '-', color='#3a7', lw=1.4, label=f'adaptive (lookahead={tr["lookahead"]})')
            ax.axvline(mig_step, ls=':', color='gray', alpha=0.7,
                       label=f'migration → {tr["migration_steps"][0]["to_tier"]}')
            ax.set_xlabel('Decode step #')
            ax.set_ylabel('Step latency (ms)')
            ax.set_title(f'{tr["model"]}: tail static={s_arr.max():.1f} ms → adaptive={a_arr.max():.1f} ms')
            ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=7)
        else:
            ax.text(0.5, 0.5, 'no migration', ha='center', va='center')
    fig.suptitle('Static vs adaptive KV-placement around the HBM→DRAM crossing',
                 fontsize=10, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig8_adaptive_vs_static.png', bbox_inches='tight')
    plt.show()
else:
    print('no adaptive trace files found — was the adaptive analysis cell run?')

## 11 · Bundle outputs for the paper

Zips `track_b_outputs/` and triggers a Colab download. **Send me this zip**, or push to your GitHub repo so I can pull figures into the paper.

In [ ]:
import shutil
ZIP = f'/content/track_b_outputs_{GPU_TAG}.zip'
if os.path.exists(ZIP): os.remove(ZIP)
shutil.make_archive(f'/content/track_b_outputs_{GPU_TAG}', 'zip', OUT_DIR)
print(f'✓ {ZIP}  ({os.path.getsize(ZIP)/1e6:.1f} MB)')

from google.colab import files
files.download(ZIP)

---

**End of notebook.** Outputs in `/content/track_b_outputs_<gpu>/`:
- `results.csv` — measured runs (model × ctx × prompt × seed)
- `sim_sweep.csv` — simulator-only context sweep (long contexts, multiple kv-bits)
- `optimizations.csv` — bytes/token under each optimization
- `compute_vs_memory.csv` — bonus: prediction error of compute-only vs memory-bound
- `adaptive_vs_static.csv` — adaptive policy headline metrics
- `adaptive_trace_*.json` — per-step latency traces under static and adaptive policies
- `figures/` — 8 PNGs for the paper

**Don't forget to run this notebook on a second GPU class** (e.g., T4 if you ran on A100 first) to enable the cross-bandwidth validation in the paper.